# 05：三层记忆协作与 SQLite 恢复
把每轮调用前的召回、输入预算、工作状态更新，与阶段结束后的归档连接起来。
两个独立 SQLite 文件分别保存 State 和跨会话记忆。每次完整运行使用新实验目录，避免旧数据干扰；
重启内核恢复时，把 RUN_DIR 设置为上次打印的相对目录，只运行恢复部分，不重跑写入。


In [ ]:
import os
import json
from pathlib import Path
from uuid import uuid4
from context_lab import (
    initialize, DemoModel, live_model, dumps, estimate_messages, request_cost,
    WorkingState, Context, archive, read_archive, summary_middleware,
    demo_summary, valid_tool_pairs, recent_messages, SUMMARY_PROMPT,
)
from langchain.agents import create_agent
from langchain.agents.middleware import wrap_model_call
from langchain.messages import HumanMessage, AIMessage, SystemMessage, ToolMessage
from langgraph.checkpoint.memory import InMemorySaver

DATA = initialize()
# 默认不调用外部模型。可在启动内核前设置 CH09_RUN_LIVE=1，或在此改为 True。
RUN_LIVE = os.getenv('CH09_RUN_LIVE', '0') == '1'
print('真实模型演示：', RUN_LIVE)


In [ ]:
from context_lab import (Episode, Candidate, namespace, core_namespace, utc_now, save_episode,
    stage_candidate, confirm_candidate, recall_episodes, load_core, all_items,
    context_middleware, BudgetExceeded, ask, make_file_tools)
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.sqlite import SqliteStore

RUN_DIR = DATA / ('integrated-' + uuid4().hex[:10])
RUN_DIR.mkdir()
print('恢复目录：', RUN_DIR)
ctx = Context('alice', 'login-lab')
thread_id = 'login-main'
audit = []

def build_agent(checkpointer, store, model, audit, real_summary=False):
    _, read_fragment, search_file, offload = make_file_tools(RUN_DIR)
    return create_agent(model, tools=[read_fragment, search_file],
        state_schema=WorkingState, context_schema=Context, store=store, checkpointer=checkpointer,
        middleware=[offload, summary_middleware(model=model if real_summary else None, trigger=5000, keep=4),
                    context_middleware(audit)])


## 1. 写入阶段：三轮需求更正，显式确认长期约定
应用负责提供可信用户、项目和 thread，示例同时记录 thread 归属。
`task` 是显式任务状态，不由这段代码自动理解自然语言生成；模型只负责回答。


In [ ]:
with SqliteSaver.from_conn_string(str(RUN_DIR / 'working.sqlite')) as saver,      SqliteStore.from_conn_string(str(RUN_DIR / 'memories.sqlite')) as store:
    saver.setup()
    store.setup()
    agent = build_agent(saver, store, DemoModel(), audit)
    task = {'goal': '登录设计', 'requirement': 'REQ-009', 'methods': ['待确认']}
    ask(agent, store, RUN_DIR, ctx, thread_id, '设计 REQ-009 登录，演示使用 SQLite。', task)
    task = {**task, 'methods': ['手机号验证码', '密码']}
    ask(agent, store, RUN_DIR, ctx, thread_id, '增加手机号验证码登录。', task)
    task = {**task, 'methods': ['手机号验证码'], 'deferred': ['密码登录'],
            'todo': ['发送频率尚未确认']}
    final = ask(agent, store, RUN_DIR, ctx, thread_id, '密码登录暂缓，发送频率尚未确认。', task)
    source = read_archive(RUN_DIR, ctx, thread_id)
    users = [m for m in source if isinstance(m, HumanMessage)]
    episode = Episode(event_id='REQ-009-design', thread_id=thread_id, time=utc_now(),
        topic='登录设计', tags=['登录', '验证码'], background='REQ-009 登录功能设计',
        decisions=['手机号验证码登录', '密码登录暂缓'], outcome='发送频率尚未确认',
        source_ids=[m.id for m in users])
    save_episode(store, ctx, episode, source)
    save_episode(store, ctx, episode, source)
    assert len(all_items(store, namespace(ctx, 'episodes'))) == 1
    candidate = Candidate(fact_key='demo_database', value='SQLite', source_ids=[users[0].id])
    cid = stage_candidate(store, ctx, candidate, source)
    confirmation = HumanMessage('确认长期约定[project]：demo_database=SQLite', id=uuid4().hex)
    archive(RUN_DIR, ctx, thread_id, [confirmation])
    confirm_candidate(store, ctx, cid, confirmation)
    assert final['task']['methods'] == ['手机号验证码']
    print('归档结果：', episode.event_id, '核心数据：', load_core(store, ctx))

print('连接已关闭；下面重新创建连接和 Agent。')


## 2. 恢复阶段：相同 thread 恢复工作状态，新 thread 召回长期信息
跨会话读取检查实际输入的来源标识和 Store 数据，不以“我记得”作为证据。
新会话不会自动继承旧任务状态；历史经历提供背景，是否继续旧任务须显式决定。


In [ ]:
with SqliteSaver.from_conn_string(str(RUN_DIR / 'working.sqlite')) as saver,      SqliteStore.from_conn_string(str(RUN_DIR / 'memories.sqlite')) as store:
    saver.setup()
    store.setup()
    agent = build_agent(saver, store, DemoModel(), audit)
    old_config = {'configurable': {'thread_id': thread_id}}
    restored = agent.get_state(old_config).values
    assert restored['task']['methods'] == ['手机号验证码']
    assert restored['task']['todo'] == ['发送频率尚未确认']
    continued = ask(agent, store, RUN_DIR, ctx, thread_id, '继续当前登录任务，下一步是什么？')
    assert continued['task'] == restored['task']
    fresh_id = 'new-' + uuid4().hex[:8]
    fresh = ask(agent, store, RUN_DIR, ctx, fresh_id, '之前登录项目有哪些决定？演示数据库是什么？', {})
    assert len(fresh['messages']) == 2
    assert fresh['task'] == {}
    assert 'REQ-009-design' in audit[-1]['episode_ids']
    assert 'demo_database' in audit[-1]['core_keys']
    assert '密码登录暂缓' in audit[-1]['prompt'] and 'SQLite' in audit[-1]['prompt']
    for other in [Context('bob', 'login-lab'), Context('alice', 'other-project')]:
        assert recall_episodes(store, other, ['登录']) == []
        assert load_core(store, other) == []
        ask(agent, store, RUN_DIR, other, 'isolated-' + uuid4().hex, '登录项目有什么约定？', {})
        assert audit[-1]['episode_ids'] == [] and audit[-1]['core_keys'] == []
    try:
        ask(agent, store, RUN_DIR, Context('bob', 'login-lab'), thread_id, '读取旧会话')
    except ValueError as exc:
        assert 'thread' in str(exc)
    else:
        raise AssertionError('不应允许复用他人 thread')
    print('通过：关闭重连、相同 thread 恢复、新 thread 召回、用户及项目隔离。')
    print('新会话回复：', fresh['messages'][-1].content)


## 3. 预算不够时，先减少低相关经历；保护当前输入和任务约束
预算计算包含系统指令、记忆、历史、工具 schema 与输出预留。
这是可观察的近似控制，实际模型 tokenizer、供应商消息封装仍可能有差异；生产需要供应商计数和余量。


In [ ]:
with SqliteSaver.from_conn_string(str(RUN_DIR / 'working.sqlite')) as saver,      SqliteStore.from_conn_string(str(RUN_DIR / 'memories.sqlite')) as store:
    saver.setup()
    store.setup()
    guard = create_agent(DemoModel(), state_schema=WorkingState, context_schema=Context,
        store=store, checkpointer=saver, middleware=[context_middleware(audit, window=3000, reserve=1000)])
    episode_count = len(all_items(store, namespace(ctx, 'episodes')))
    try:
        ask(guard, store, RUN_DIR, ctx, 'oversize', '本轮必须保留的输入' * 3000,
            {'goal': '登录', 'constraint': '当前输入不得静默截断'})
    except BudgetExceeded as exc:
        print('预期停止：', exc)
    else:
        raise AssertionError('超预算应停止')
    assert len(all_items(store, namespace(ctx, 'episodes'))) == episode_count
    assert len(read_archive(RUN_DIR, ctx, 'oversize')) == 1
    print('最近调用审计（不输出整段原文）：')
    print({k: v for k, v in audit[-1].items() if k not in {'sent_messages', 'prompt'}})
    # 预算缩减的两个成功分支：先去掉低相关经历，再裁剪旧轮次；当前任务不丢。
    crowded = Episode(event_id='crowded', thread_id=thread_id, time=utc_now(), topic='登录背景',
        tags=['登录'], background='资料' * 240, decisions=['手机号验证码'],
        outcome='背景详情' * 160, source_ids=[users[0].id])
    save_episode(store, ctx, crowded, read_archive(RUN_DIR, ctx, thread_id))
    small_audit = []
    small = create_agent(DemoModel(), state_schema=WorkingState, context_schema=Context, store=store,
                         middleware=[context_middleware(small_audit, window=3000, reserve=1000)])
    small.invoke({'messages': [HumanMessage('登录')], 'task': task}, context=ctx)
    assert 'crowded' not in small_audit[-1]['episode_ids']
    assert small_audit[-1]['input_estimate'] + 1000 <= 3000
    noisy = [HumanMessage('旧布局讨论' * 1000), AIMessage('旧答复'),
             HumanMessage('无新增决定'), AIMessage('继续'),
             HumanMessage('继续核对'), AIMessage('没有变更'), HumanMessage('当前登录状态？')]
    small.invoke({'messages': noisy, 'task': task}, context=ctx)
    assert len(small_audit[-1]['sent_messages']) < len(noisy)
    assert small_audit[-1]['sent_messages'][-1]['data']['content'] == '当前登录状态？'
    assert '发送频率尚未确认' in small_audit[-1]['prompt']
    store.delete(namespace(ctx, 'episodes'), 'crowded')


## 4. 可选真实模型恢复
仅调用两轮：原会话继续、新会话读取。SQLite 已由前面的离线流程写入，
此处验证真实模型收到召回资料并生成回复，不把手工写入说成模型自动归档。


In [ ]:
if RUN_LIVE:
    with SqliteSaver.from_conn_string(str(RUN_DIR / 'working.sqlite')) as saver,          SqliteStore.from_conn_string(str(RUN_DIR / 'memories.sqlite')) as store:
        saver.setup()
        store.setup()
        real_agent = build_agent(saver, store, live_model(), audit, real_summary=True)
        one = ask(real_agent, store, RUN_DIR, ctx, thread_id, '继续登录任务，列出有效需求和未确认事项。')
        assert one['task']['methods'] == ['手机号验证码']
        two = ask(real_agent, store, RUN_DIR, ctx, 'real-new-' + uuid4().hex[:8],
                  '回顾之前登录设计决定和本地演示数据库约定。', {})
        assert audit[-1]['episode_ids'] == ['REQ-009-design']
        assert 'demo_database' in audit[-1]['core_keys']
        assert store.get(namespace(ctx, 'core'), 'demo_database').value['value'] == 'SQLite'
        print(one['messages'][-1].content)
        print(two['messages'][-1].content)
else:
    print('未执行真实模型恢复。')


## 结果解释
工作记忆由 Checkpointer 管理；情景和核心记忆由 Store 管理。
摘要、文件卸载控制工作上下文；情景归档与核心确认是另外的写入步骤。
当前实现是单进程教学实验，thread 归属记录不能替代生产身份认证与并发事务。
